# Model Prep Notebook

This notebook goes over the features.csv final dataset and gets it ready for the modeling phase by dropping unnecessary columns, adding prediction outcomes, and doubling the rows for winners and losers.

## 1. Load final features dataset

In [11]:
import pandas as pd

df = pd.read_csv("../data/processed/features.csv", low_memory=False)
print(df.shape)
df.head()

(24875, 142)


,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,Winner,...,loser_ytd_Return_Points_Won,winner_ytd_Total_Points_Won,loser_ytd_Total_Points_Won,location,year,winner_match_,loser_match_,winner_form,loser_form,form_diff
0,1,Brisbane,Brisbane International,2015-01-05,ATP250,Outdoor,Hard,1st Round,3.0,Chardy J.,...,33%,51%,47%,Brisbane,2015.0,NaN,NaN,NaN,NaN,NaN
1,2,Chennai,Chennai Open,2015-01-05,ATP250,Outdoor,Hard,1st Round,3.0,Muller G.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,Chennai,Chennai Open,2015-01-05,ATP250,Outdoor,Hard,1st Round,3.0,Haider-Maurer A.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,Chennai,Chennai Open,2015-01-05,ATP250,Outdoor,Hard,1st Round,3.0,Coric B.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,Doha,Qatar Exxon Mobil Open,2015-01-05,ATP250,Outdoor,Hard,1st Round,3.0,Bolelli S.,...,33%,50%,48%,Doha,2015.0,NaN,NaN,NaN,NaN,NaN


## 2. Drop non-modeling columns

We remove text labels / metadata:

Tournament

Round

Series

Location

Comment

dates (unless using as feature)

As well as any post-match columns that might give away the result. 

In [12]:
columns_to_drop = {
    'Duplicate columns': ['winner', 'location', 'year', 'round', 'Year'],
    'Empty columns': ['winner_match_', 'loser_match_'],
    'Match statistics (DATA LEAKAGE)': [
        col for col in df.columns if 'match_' in col and 
        any(x in col for x in ['Serve_Rating', 'Return_Rating', 'Aces', 'Double_Faults',
                               '1st_Serve', '2nd_Serve', 'Points_Won', 'Break_Points',
                               'Service_Games', 'Return_Games', 'Service_Points',
                               'Return_Points', 'Total_Points'])
    ],
    'Match outcome indicators': ['Wsets', 'Lsets', 'higher_elo_won'],
    'Redundant identifiers': ['player1', 'player2', 'seed1', 'seed2', 'scores1', 'scores2', 'Loser_player_id', 'Winner_player_id'],
    'Bookmaker-specific odds': [
        'B365W', 'B365L', 'PSW', 'PSL', 'B365_W_ip', 'B365_L_ip',
        'B365_overround', 'B365_W_fair', 'B365_L_fair',
        'PS_W_ip', 'PS_L_ip', 'PS_overround', 'PS_W_fair', 'PS_L_fair',
        'winner_odds','loser_odds','winner_implied_prob','loser_implied_prob'
    ],
    'User-specified drops': ['overround', 'arb_flag', 'stats_url', 'duration', 'h2h', 'Loser_ioc', 'Winner_ioc']
}

all_drops = []
for category, cols in columns_to_drop.items():
    existing_cols = [c for c in cols if c in df.columns]
    all_drops.extend(existing_cols)
    print(f"\n{category}: {len(existing_cols)} columns")

print(f"TOTAL COLUMNS TO DROP: {len(all_drops)}")

# Drop columns
df_clean = df.drop(columns=all_drops)
print(f"Remaining columns: {len(df_clean.columns)}\n")


Duplicate columns: 5 columns

Empty columns: 2 columns

Match statistics (DATA LEAKAGE): 32 columns

Match outcome indicators: 3 columns

Redundant identifiers: 8 columns

Bookmaker-specific odds: 18 columns

User-specified drops: 7 columns
TOTAL COLUMNS TO DROP: 75
Remaining columns: 67



## **CRITICAL:** 3. Convert the dataset to a two-row-per-match format

This is **required** for any supervised ML classifier.

For each match:

**Row A** → Winner perspective → WinnerBinary = 1

**Row B** → Loser perspective → WinnerBinary = 0
(swap all winner/loser features)

In [13]:
from pathlib import Path

# Version 1: WITHOUT YTD stats
ytd_cols = [col for col in df.columns if 'ytd' in col.lower()]
print(f"\nYTD columns to drop: {len(ytd_cols)}")

df_no_ytd = df_clean.copy()
df_no_ytd = df_no_ytd.drop(columns=ytd_cols)

# Version 2: WITH YTD stats (but drop the specified columns)
df_with_ytd = df_clean.copy()

print(f"\nVersion 1 (no YTD): {df_no_ytd.shape}")
print(f"Version 2 (with YTD): {df_with_ytd.shape}")

# ============================================================================
# SAVE BOTH VERSIONS
# ============================================================================

out_dir = Path("../data/processed/model_ready")
out_dir.mkdir(parents=True, exist_ok=True)

output_no_ytd = out_dir / "features_no_ytd.csv"
output_with_ytd = out_dir / "features_with_ytd.csv"

df_no_ytd.to_csv(output_no_ytd, index=False)
df_with_ytd.to_csv(output_with_ytd, index=False)

print(f"\nSAVED: {output_no_ytd}")
print(f"SAVED: {output_with_ytd}")


YTD columns to drop: 36

Version 1 (no YTD): (24875, 31)
Version 2 (with YTD): (24875, 67)

SAVED: ../data/processed/model_ready/features_no_ytd.csv
SAVED: ../data/processed/model_ready/features_with_ytd.csv
